In [52]:
"""Vector store module for semantic search in E-commerce Product Search System.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from product content using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique product identifier.
    - 'title': The product title.
    - 'description': The product description (used for embeddings).
    - 'category': Product category.
    - 'price': Product price.
    - 'rating': Average rating.
    - 'review_count': Review count.

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.
    Embeddings should be generated from product content (title + description).

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    Implements brute-force approach: compute similarity with all vectors, then return top-k.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
                Default: "all-MiniLM-L6-v2" (384 dimensions)
        """
        self.model_name = model_name
        self.model = None

    def _ensure_model_loaded(self):
        """Lazy load the SentenceTransformer model."""
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., product titles + descriptions).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).
                For all-MiniLM-L6-v2, embedding_dim = 384.

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        # Validate input
        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        if len(texts) == 0:
            raise ValueError("texts list cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embeddings
        embeddings = self.model.encode(texts, convert_to_numpy=True)

        return embeddings

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for product content (title + description).
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing products with required fields:
                - 'id': Unique product ID.
                - 'title': Product title.
                - 'description': Product description.
                - 'category': Product category.
                - 'price': Product price.
                - 'rating': Average rating.
                - 'review_count': Review count.
            index_file_name (str): Name of the index file (e.g., 'product_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing:
                - 'embeddings': numpy array of embeddings (shape: [n_docs, 384])
                - 'id': list of product IDs
                - 'title': list of titles
                - 'description': list of descriptions
                - 'category': list of categories
                - 'price': list of prices
                - 'rating': list of ratings
                - 'review_count': list of review counts

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        # Validate DataFrame
        if not isinstance(df, pd.DataFrame):
            raise ValueError("df must be a pandas DataFrame")

        if len(df) == 0:
            raise ValueError("DataFrame is empty")

        # Validate required columns
        required_columns = ['id', 'title', 'description', 'category', 'price', 'rating', 'review_count']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        # Generate embeddings from product content (title + description)
        texts = [f"{row['title']}\n\n{row['description']}" for _, row in df.iterrows()]
        embeddings = self.generate_embeddings(texts)

        # Create index dictionary
        index = {
            'embeddings': embeddings,
            'id': df['id'].tolist(),
            'title': df['title'].tolist(),
            'description': df['description'].tolist(),
            'category': df['category'].tolist(),
            'price': df['price'].tolist(),
            'rating': df['rating'].tolist(),
            'review_count': df['review_count'].tolist()
        }

        # Create directory if it doesn't exist
        index_folder = Path(index_folder_name)
        index_folder.mkdir(parents=True, exist_ok=True)

        # Save index to disk
        index_path = index_folder / index_file_name
        with open(index_path, 'wb') as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'product_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata (same structure as create_index).

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        # Construct full path
        index_path = Path(index_folder_name) / index_file_name

        # Check if file exists
        if not index_path.exists():
            raise FileNotFoundError(f"Index file not found: {index_path}")

        # Load index
        try:
            with open(index_path, 'rb') as f:
                index = pickle.load(f)
        except Exception as e:
            raise ValueError(f"Failed to load index: {str(e)}")

        # Validate index structure
        required_keys = ['embeddings', 'id', 'title', 'description', 'category', 'price', 'rating', 'review_count']
        missing_keys = [key for key in required_keys if key not in index]
        if missing_keys:
            raise KeyError(f"Index missing required keys: {missing_keys}")

        # Validate embeddings shape
        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        if len(index['embeddings']) == 0:
            raise ValueError("Index embeddings array is empty")

        # Validate metadata lengths match
        n_docs = len(index['embeddings'])
        for key in required_keys:
            if len(index[key]) != n_docs:
                raise ValueError(f"Index metadata length mismatch: {key} has {len(index[key])} items, expected {n_docs}")

        return index

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).
                For all-MiniLM-L6-v2, shape = (384,).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        # Validate input
        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not query.strip():
            raise ValueError("query cannot be empty")

        # Load model if needed
        self._ensure_model_loaded()

        # Generate embedding (single text, returns 1D array)
        embedding = self.model.encode(query, convert_to_numpy=True)

        # Ensure it's 1D
        if embedding.ndim > 1:
            embedding = embedding[0]

        return embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar entries from the index using cosine similarity.

        Implementation uses brute-force approach:
        1. Compute cosine similarity between query_embedding and all embeddings in index
        2. Sort results by similarity score (descending)
        3. Return top-k matches

        Cosine similarity formula:
        similarity = dot(a, b) / (norm(a) * norm(b))

        Args:
            query_embedding (np.ndarray): The embedding of the input query (shape: [384]).
            index (dict): The index containing document embeddings and metadata.
                Expected structure: {'embeddings': np.ndarray, 'id': list, ...}
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.
                  - document_index: Index position in the original DataFrame/index
                  - similarity_score: Cosine similarity score (float, typically 0.0 to 1.0)

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        # Validate inputs
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("query_embedding must be a numpy array")

        if 'embeddings' not in index:
            raise ValueError("Index must contain 'embeddings' key")

        if not isinstance(index['embeddings'], np.ndarray):
            raise ValueError("Index embeddings must be a numpy array")

        # Get document embeddings
        doc_embeddings = index['embeddings']

        # Validate shapes
        if query_embedding.ndim != 1:
            raise ValueError("query_embedding must be 1D array")

        if doc_embeddings.ndim != 2:
            raise ValueError("doc_embeddings must be 2D array")

        if query_embedding.shape[0] != doc_embeddings.shape[1]:
            raise ValueError(f"Dimension mismatch: query has {query_embedding.shape[0]} dims, docs have {doc_embeddings.shape[1]} dims")

        # Compute cosine similarity for all documents (brute-force)
        # Cosine similarity = dot(a, b) / (norm(a) * norm(b))
        query_norm = np.linalg.norm(query_embedding)
        doc_norms = np.linalg.norm(doc_embeddings, axis=1)

        # Compute dot products
        dot_products = np.dot(doc_embeddings, query_embedding)

        # Compute cosine similarities
        similarities = dot_products / (query_norm * doc_norms)

        # Handle any NaN values (shouldn't happen, but safety check)
        similarities = np.nan_to_num(similarities, nan=0.0)

        # Get top-k indices
        top_k_indices = np.argsort(similarities)[::-1][:k]

        # Create list of (index, similarity) tuples
        results = [(int(idx), float(similarities[idx])) for idx in top_k_indices]

        return results



/Users/efloresp06/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
"""HNSW (Hierarchical Navigable Small World) indexing module for scalable vector search.

This module provides functionality to build and query HNSW indexes for fast approximate
nearest neighbor search in large vector collections.
"""

import random
from typing import Dict, List, Tuple

import numpy as np


class HNSWIndex:
    """
    A simplified HNSW (Hierarchical Navigable Small World) index for approximate nearest neighbor search.

    HNSW creates a multi-layer graph where vectors are nodes connected to their nearest neighbors.
    Search starts at the top layer (sparse, long-range connections) and moves down to the bottom
    layer (dense, local connections), efficiently navigating through vector space.

    Attributes:
        max_connections (int): Maximum number of connections per node in each layer.
        ef_construction (int): Size of dynamic candidate list during construction.
        m (float): Probability parameter for layer assignment (default: 0.5).
        layers (Dict[int, Dict[int, List[int]]]): Multi-layer graph structure.
        vectors (Dict[int, np.ndarray]): Vector storage by node ID.
        entry_point (int): Highest layer entry point node ID.
    """

    def __init__(self, max_connections: int = 16, ef_construction: int = 200, m: float = 0.5):
        """
        Initializes the HNSW index with configuration parameters.

        Args:
            max_connections (int): Maximum number of connections per node (default: 16).
            ef_construction (int): Size of dynamic candidate list during construction (default: 200).
            m (float): Probability parameter for layer assignment (default: 0.5).

        Raises:
            ValueError: If parameters are invalid.
        """
        if max_connections < 1:
            raise ValueError("max_connections must be at least 1")
        if ef_construction < 1:
            raise ValueError("ef_construction must be at least 1")
        if not 0.0 < m <= 1.0:
            raise ValueError("m must be between 0.0 and 1.0")

        self.max_connections = max_connections
        self.ef_construction = ef_construction
        self.m = m
        self.layers: Dict[int, Dict[int, List[int]]] = {}  # layer -> {node_id: [neighbor_ids]}
        self.vectors: Dict[int, np.ndarray] = {}  # node_id -> vector
        self.entry_point: int = None
        self.max_layer = -1

    def _cosine_distance(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """
        Calculates cosine distance between two vectors.

        Cosine distance = 1 - cosine_similarity

        Args:
            vec1 (np.ndarray): First vector.
            vec2 (np.ndarray): Second vector.

        Returns:
            float: Cosine distance (0.0 to 2.0, where 0.0 = identical).
        """
        cosine_similarity = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
        return 1.0 - cosine_similarity

    def _assign_layer(self) -> int:
        """
        Randomly assigns a layer to a new node using exponential decay.

        Layer 0 is always assigned. Higher layers are assigned with decreasing probability.

        Returns:
            int: Layer number (0 = bottom layer, higher = top layers).
        """
        layer = 0

        while random.random() < self.m:
            layer += 1

        return layer

    def _select_neighbors(self, query_vec: np.ndarray, candidates: List[int], k: int) -> List[int]:
        """
        Selects k nearest neighbors from candidates using greedy selection.

        Args:
            query_vec (np.ndarray): Query vector.
            candidates (List[int]): Candidate node IDs.
            k (int): Number of neighbors to select.

        Returns:
            List[int]: List of k nearest neighbor node IDs, sorted by distance (ascending).
        """
        if k <= 0:
            return []

        distances = [(node_id, self._cosine_distance(query_vec, self.vectors[node_id])) for node_id in candidates]
        distances.sort(key=lambda x: x[1])  # Sort by distance

        return [node_id for node_id, _ in distances[:k]]

    def _search_layer(self, query_vec: np.ndarray, layer: int, ef: int, entry_nodes: List[int]) -> List[int]:
        """
        Searches a single layer for nearest neighbors.

        Uses greedy search starting from entry_nodes, exploring up to ef candidates.

        Args:
            query_vec (np.ndarray): Query vector.
            layer (int): Layer number to search.
            ef (int): Number of candidates to explore (search_ef parameter).
            entry_nodes (List[int]): Starting node IDs for search.

        Returns:
            List[int]: List of candidate node IDs found in this layer, sorted by distance.
        """
        visited = set(entry_nodes)
        candidates = set(entry_nodes)

        results = []

        while candidates:

            current = min(
                candidates,
                key=lambda n: self._cosine_distance(
                    query_vec,
                    self.vectors[n]
                )
            )

            candidates.remove(current)

            current_distance = self._cosine_distance(query_vec, self.vectors[current])
            results.append((current, current_distance))

            for neighbor in self.layers.get(layer, {}).get(current, []):

                if neighbor in visited:
                    continue

                visited.add(neighbor)
                candidates.add(neighbor)

        results.sort(key=lambda x: x[1])  # Sort by distance
        return [node_id for node_id, _ in results[:ef]]

    def build_index(self, vectors: np.ndarray) -> None:
        """
        Builds HNSW index from a collection of vectors.

        Process:
        1. For each vector, assign it to layers (0 to assigned_layer)
        2. For each layer, find nearest neighbors and create connections
        3. Update entry_point if this is the highest layer node

        Args:
            vectors (np.ndarray): Array of vectors to index (shape: [n_vectors, embedding_dim]).

        Raises:
            ValueError: If vectors array is empty or invalid.
            TypeError: If vectors is not a numpy array.
        """
        if not isinstance(vectors, np.ndarray):
            raise TypeError("vectors must be a numpy array")

        if vectors.shape[0] == 0:
            raise ValueError("vectors array is empty")

        if vectors.ndim != 2:
            raise ValueError("vectors must be a 2D array")

        for vector in vectors:

            node = len(self.vectors)
            level = self._assign_layer()
            self.vectors[node] = vector

            if self.entry_point is None:
                for layer in range(level + 1):
                    self.layers.setdefault(layer, {})
                    self.layers[layer][node] = []

                self.entry_point = node
                self.max_layer = level
                continue

            current = self.entry_point

            for layer in range(
                self.max_layer,
                level,
                -1
            ):
                current = self._search_layer(
                    vector,
                    layer,
                    1,
                    [current]
                )[0]

            for layer in range(
                min(level, self.max_layer)
            ):
                candidates = self._search_layer(
                    vector,
                    layer,
                    self.ef_construction,
                    [current]
                )

                neighbors = self._select_neighbors(
                    vector,
                    candidates,
                    self.max_connections
                )

                self.layers.setdefault(layer, {})
                self.layers[layer][node] = neighbors

                for neighbor in neighbors:
                    self.layers[layer].setdefault(neighbor, [])
                    self.layers[layer][neighbor].append(node)

                    if len(self.layers[layer][neighbor]) > self.max_connections:

                        self.layers[layer][neighbor] = self._select_neighbors(
                            self.vectors[neighbor],
                            self.layers[layer][neighbor],
                            self.max_connections
                        )

                if candidates:
                    current = candidates[0]

            if level > self.max_layer:
                self.max_layer = level
                self.entry_point = node

    def search(self, query_vec: np.ndarray, k: int = 5, search_ef: int = 50) -> List[Tuple[int, float]]:
        """
        Searches the HNSW index for k nearest neighbors.

        Process:
        1. Start from entry_point at top layer
        2. Search each layer from top to bottom
        3. At bottom layer (layer 0), return top-k results

        Args:
            query_vec (np.ndarray): Query vector (shape: [embedding_dim]).
            k (int): Number of nearest neighbors to return (default: 5).
            search_ef (int): Number of candidates to explore (default: 50).
                Higher values = better recall but slower search.

        Returns:
            List[Tuple[int, float]]: List of (node_id, distance) tuples for k nearest neighbors,
                                     sorted by distance (ascending).

        Raises:
            ValueError: If index is empty or query_vec is invalid.
            TypeError: If query_vec is not a numpy array.
        """
        if not self.vectors:
            raise ValueError("HNSW index is empty. Build the index before searching.")

        if not isinstance(query_vec, np.ndarray):
            raise TypeError("query_vec must be a numpy array")

        if query_vec.ndim != 1:
            raise ValueError("query_vec must be a 1D array")

        if query_vec.shape[0] != next(iter(self.vectors.values())).shape[0]:
            raise ValueError("query_vec dimension does not match indexed vectors")

        if search_ef < k:
            raise ValueError("search_ef must be greater than or equal to k")

        if k <= 0:
            raise ValueError("k must be a positive integer")

        current = self.entry_point

        for layer in range(
            self.max_layer,
            0,
            -1
        ):
            current = self._search_layer(
                query_vec,
                layer,
                1,
                [current]
            )[0]

        candidates = self._search_layer(
            query_vec,
            0,
            search_ef,
            [current]
        )

        neighbors = self._select_neighbors(
            query_vec,
            candidates,
            k
        )

        return [(node_id, float(self._cosine_distance(query_vec, self.vectors[node_id]))) for node_id in neighbors]

    def get_vector(self, node_id: int) -> np.ndarray:
        """
        Retrieves a vector by node ID.

        Args:
            node_id (int): Node ID.

        Returns:
            np.ndarray: Vector associated with the node ID.

        Raises:
            KeyError: If node_id does not exist.
        """
        if node_id not in self.vectors:
            raise KeyError(f"Node ID {node_id} not found in index")
        return self.vectors[node_id]

In [47]:
"""Multi-level caching module for RAG system performance optimization.

This module provides functionality to implement three-tier caching:
1. Embedding cache: Cache query embeddings
2. Search result cache: Cache top-K search results
3. Response cache: Cache final generated responses
"""

import hashlib
import time
from collections import OrderedDict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


class CacheManager:
    """
    Manages multi-level caching for RAG system performance optimization.

    Implements three cache levels:
    - Embedding cache: Stores query -> embedding mappings
    - Search result cache: Stores (query, k) -> top-k results mappings
    - Response cache: Stores query -> final response mappings

    Uses LRU (Least Recently Used) eviction policy when cache limits are reached.

    Attributes:
        embedding_cache_size (int): Maximum number of embeddings to cache.
        search_cache_size (int): Maximum number of search results to cache.
        response_cache_size (int): Maximum number of responses to cache.
        embedding_ttl (float): Time-to-live for embedding cache entries (seconds).
        search_ttl (float): Time-to-live for search cache entries (seconds).
        response_ttl (float): Time-to-live for response cache entries (seconds).
    """

    def __init__(
        self,
        embedding_cache_size: int = 1000,
        search_cache_size: int = 500,
        response_cache_size: int = 200,
        embedding_ttl: float = 3600.0,
        search_ttl: float = 1800.0,
        response_ttl: float = 600.0
    ):
        """
        Initializes the CacheManager with configurable cache sizes and TTLs.

        Args:
            embedding_cache_size (int): Maximum embeddings to cache (default: 1000).
            search_cache_size (int): Maximum search results to cache (default: 500).
            response_cache_size (int): Maximum responses to cache (default: 200).
            embedding_ttl (float): Embedding cache TTL in seconds (default: 3600).
            search_ttl (float): Search cache TTL in seconds (default: 1800).
            response_ttl (float): Response cache TTL in seconds (default: 600).

        Raises:
            ValueError: If cache sizes or TTLs are invalid.
        """
        if embedding_cache_size < 1:
            raise ValueError("embedding_cache_size must be at least 1")
        if search_cache_size < 1:
            raise ValueError("search_cache_size must be at least 1")
        if response_cache_size < 1:
            raise ValueError("response_cache_size must be at least 1")
        if embedding_ttl <= 0:
            raise ValueError("embedding_ttl must be positive")
        if search_ttl <= 0:
            raise ValueError("search_ttl must be positive")
        if response_ttl <= 0:
            raise ValueError("response_ttl must be positive")

        self.embedding_cache_size = embedding_cache_size
        self.search_cache_size = search_cache_size
        self.response_cache_size = response_cache_size
        self.embedding_ttl = embedding_ttl
        self.search_ttl = search_ttl
        self.response_ttl = response_ttl

        # LRU caches: OrderedDict maintains insertion order
        self.embedding_cache: OrderedDict[str, Tuple[np.ndarray, float]] = OrderedDict()
        self.search_cache: OrderedDict[str, Tuple[List[Tuple[int, float]], float]] = OrderedDict()
        self.response_cache: OrderedDict[str, Tuple[Any, float]] = OrderedDict()

        # Cache statistics
        self.embedding_hits = 0
        self.embedding_misses = 0
        self.search_hits = 0
        self.search_misses = 0
        self.response_hits = 0
        self.response_misses = 0

    def _generate_cache_key(self, query: str, k: Optional[int] = None) -> str:
        """
        Generates a cache key from query string and optional k parameter.

        Args:
            query (str): Query string.
            k (Optional[int]): Optional k parameter for search cache.

        Returns:
            str: Cache key (hash of query + optional k).
        """
        key_str = f"{query}:{k}" if k is not None else query
        return hashlib.sha256(key_str.encode("utf-8")).hexdigest()

    def _is_expired(self, timestamp: float, ttl: float) -> bool:
        """
        Checks if a cache entry has expired.

        Args:
            timestamp (float): Entry creation timestamp.
            ttl (float): Time-to-live in seconds.

        Returns:
            bool: True if expired, False otherwise.
        """
        return (time.time() - timestamp) > ttl

    def get_embedding(self, query: str) -> Optional[np.ndarray]:
        """
        Retrieves cached embedding for a query.

        Args:
            query (str): Query string.

        Returns:
            Optional[np.ndarray]: Cached embedding if found and not expired, None otherwise.
        """
        key = self._generate_cache_key(query)
        cached = self.embedding_cache.get(key)
        if cached and not self._is_expired(cached[1], self.embedding_ttl):
            self.embedding_hits += 1
            # Move to end to mark as recently used
            self.embedding_cache.move_to_end(key)
            return cached[0]
        self.embedding_misses += 1
        return None

    def set_embedding(self, query: str, embedding: np.ndarray) -> None:
        """
        Stores embedding in cache with LRU eviction.

        Args:
            query (str): Query string.
            embedding (np.ndarray): Embedding vector to cache.
        """
        key = self._generate_cache_key(query)
        self.embedding_cache[key] = (embedding, time.time())
        self.embedding_cache.move_to_end(key)

        # LRU eviction: remove oldest entries when cache is full
        while len(self.embedding_cache) > self.embedding_cache_size:
            self.embedding_cache.popitem(last=False)

    def get_search_results(self, query: str, k: int) -> Optional[List[Tuple[int, float]]]:
        """
        Retrieves cached search results for a query and k value.

        Args:
            query (str): Query string.
            k (int): Number of results requested.

        Returns:
            Optional[List[Tuple[int, float]]]: Cached results if found and not expired, None otherwise.
        """
        key = self._generate_cache_key(query, k)
        cached = self.search_cache.get(key)
        if cached and not self._is_expired(cached[1], self.search_ttl):
            self.search_hits += 1
            # Move to end to mark as recently used
            self.search_cache.move_to_end(key)
            return cached[0]
        self.search_misses += 1
        return None

    def set_search_results(self, query: str, k: int, results: List[Tuple[int, float]]) -> None:
        """
        Stores search results in cache with LRU eviction.

        Args:
            query (str): Query string.
            k (int): Number of results.
            results (List[Tuple[int, float]]): Search results to cache.
        """
        key = self._generate_cache_key(query, k)
        self.search_cache[key] = (results, time.time())
        self.search_cache.move_to_end(key)

        # LRU eviction
        while len(self.search_cache) > self.search_cache_size:
            self.search_cache.popitem(last=False)

    def get_response(self, query: str) -> Optional[Any]:
        """
        Retrieves cached response for a query.

        Args:
            query (str): Query string.

        Returns:
            Optional[Any]: Cached response if found and not expired, None otherwise.
        """
        key = self._generate_cache_key(query)
        cached = self.response_cache.get(key)
        if cached and not self._is_expired(cached[1], self.response_ttl):
            self.response_hits += 1
            # Move to end to mark as recently used
            self.response_cache.move_to_end(key)
            return cached[0]
        self.response_misses += 1
        return None

    def set_response(self, query: str, response: Any) -> None:
        """
        Stores response in cache with LRU eviction.

        Args:
            query (str): Query string.
            response (Any): Response to cache.
        """
        key = self._generate_cache_key(query)
        self.response_cache[key] = (response, time.time())
        self.response_cache.move_to_end(key)

        # LRU eviction
        while len(self.response_cache) > self.response_cache_size:
            self.response_cache.popitem(last=False)

    def get_cache_stats(self) -> Dict[str, float]:
        """
        Returns cache hit rate statistics for all cache levels.

        Returns:
            Dict[str, float]: Dictionary with hit rates:
                - 'embedding_hit_rate': Hit rate for embedding cache
                - 'search_hit_rate': Hit rate for search cache
                - 'response_hit_rate': Hit rate for response cache
        """
        embedding_total = self.embedding_hits + self.embedding_misses
        search_total = self.search_hits + self.search_misses
        response_total = self.response_hits + self.response_misses

        return {
            "embedding_hit_rate": self.embedding_hits / embedding_total if embedding_total > 0 else 0.0,
            "search_hit_rate": self.search_hits / search_total if search_total > 0 else 0.0,
            "response_hit_rate": self.response_hits / response_total if response_total > 0 else 0.0,
        }

    def clear_cache(self, cache_type: Optional[str] = None) -> None:
        """
        Clears cache(s) and resets statistics.

        Args:
            cache_type (Optional[str]): Type of cache to clear ('embedding', 'search', 'response'),
                                        or None to clear all caches.
        """
        if cache_type not in {None, "embedding", "search", "response"}:
            raise ValueError(f"Invalid cache_type: {cache_type}")

        if cache_type == "embedding":
            self.embedding_cache.clear()
            self.embedding_hits = 0
            self.embedding_misses = 0

        elif cache_type == "search":
            self.search_cache.clear()
            self.search_hits = 0
            self.search_misses = 0

        elif cache_type == "response":
            self.response_cache.clear()
            self.response_hits = 0
            self.response_misses = 0

        else:
            self.embedding_cache.clear()
            self.search_cache.clear()
            self.response_cache.clear()
            self.embedding_hits = 0
            self.embedding_misses = 0
            self.search_hits = 0
            self.search_misses = 0
            self.response_hits = 0
            self.response_misses = 0

In [48]:
"""Metrics collection module for RAG system monitoring and evaluation.

This module provides functionality to track performance metrics including:
- Precision@K and Recall@K
- Cache hit rates
- Query latency
- Performance statistics over time
"""

import time
import statistics
from collections import deque
from dataclasses import dataclass
from datetime import datetime
from datetime import timedelta
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


@dataclass
class QueryMetrics:
    """Metrics for a single query execution."""

    query_id: str
    timestamp: datetime
    total_latency_ms: float
    embedding_latency_ms: float
    search_latency_ms: float
    retrieved_count: int
    cache_hits: Dict[str, bool]  # {'embedding': bool, 'search': bool, 'response': bool}
    error_type: Optional[str] = None


class MetricsCollector:
    """
    Collects and analyzes performance metrics for RAG system monitoring.

    Tracks:
    - Query latency (total and component breakdown)
    - Cache hit rates
    - Precision@K and Recall@K (when ground truth is available)
    - Performance statistics over time

    Attributes:
        retention_hours (int): Number of hours to retain metrics (default: 24).
        metrics_history (deque): History of query metrics.
    """

    def __init__(self, retention_hours: int = 24):
        """
        Initializes the MetricsCollector.

        Args:
            retention_hours (int): Number of hours to retain metrics (default: 24).

        Raises:
            ValueError: If retention_hours is invalid.
        """
        if retention_hours < 1:
            raise ValueError("retention_hours must be at least 1")

        self.retention_hours = retention_hours
        self.metrics_history: deque = deque()

    def record_query(self, metrics: QueryMetrics) -> None:
        """
        Records metrics for a single query.

        Args:
            metrics (QueryMetrics): Query metrics to record.
        """
        self.metrics_history.append(metrics)
        self._cleanup_old_data()

    def _cleanup_old_data(self) -> None:
        """
        Removes metrics older than retention_hours.
        """
        cutoff_time = datetime.now() - timedelta(hours=self.retention_hours)
        self.metrics_history = deque(
            m for m in self.metrics_history if m.timestamp > cutoff_time
        )

    def calculate_precision_at_k(self, retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
        """
        Calculates Precision@K metric.

        Precision@K = (Relevant documents in top K) / K

        Args:
            retrieved_ids (List[int]): List of retrieved document IDs (top K).
            relevant_ids (List[int]): List of all relevant document IDs.
            k (int): Value of K for Precision@K.

        Returns:
            float: Precision@K score (0.0 to 1.0).
        """
        if k == 0:
            return 0.0

        relevant_retrieved = len(set(retrieved_ids[:k]) & set(relevant_ids))
        return relevant_retrieved / k

    def calculate_recall_at_k(self, retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
        """
        Calculates Recall@K metric.

        Recall@K = (Relevant documents in top K) / (Total relevant documents)

        Args:
            retrieved_ids (List[int]): List of retrieved document IDs (top K).
            relevant_ids (List[int]): List of all relevant document IDs.
            k (int): Value of K for Recall@K.

        Returns:
            float: Recall@K score (0.0 to 1.0).
        """
        if not relevant_ids:
            return 0.0

        relevant_retrieved = len(set(retrieved_ids[:k]) & set(relevant_ids))
        return relevant_retrieved / len(relevant_ids)

    def get_cache_hit_rates(self) -> Dict[str, float]:
        """
        Calculates cache hit rates from recent metrics.

        Returns:
            Dict[str, float]: Dictionary with hit rates:
                - 'embedding_hit_rate': Hit rate for embedding cache
                - 'search_hit_rate': Hit rate for search cache
                - 'response_hit_rate': Hit rate for response cache
        """
        cache_hits = {
            "embedding_hit_rate": 0,
            "search_hit_rate": 0,
            "response_hit_rate": 0
        }

        total_queries = len(self.metrics_history)

        for metrics in self.metrics_history:
            cache_hits["embedding_hit_rate"] += metrics.cache_hits.get("embedding", False)
            cache_hits["search_hit_rate"] += metrics.cache_hits.get("search", False)
            cache_hits["response_hit_rate"] += metrics.cache_hits.get("response", False)

        if total_queries > 0:
            for key in cache_hits:
                cache_hits[key] /= total_queries

        return cache_hits

    def get_latency_stats(self) -> Dict[str, float]:
        """
        Calculates latency statistics from recent metrics.

        Returns:
            Dict[str, float]: Dictionary with statistics:
                - 'mean_total_latency_ms': Mean total latency
                - 'mean_embedding_latency_ms': Mean embedding latency
                - 'mean_search_latency_ms': Mean search latency
                - 'p95_total_latency_ms': 95th percentile total latency
        """
        total_latencies = [m.total_latency_ms for m in self.metrics_history]
        embedding_latencies = [m.embedding_latency_ms for m in self.metrics_history]
        search_latencies = [m.search_latency_ms for m in self.metrics_history]

        # Calculate p95 using numpy.percentile for more intuitive results
        # numpy.percentile uses linear interpolation by default
        if len(total_latencies) >= 1:
            p95_latency = float(np.percentile(total_latencies, 95))
        else:
            p95_latency = 0.0

        stats = {
            "mean_total_latency_ms": statistics.mean(total_latencies) if total_latencies else 0.0,
            "mean_embedding_latency_ms": statistics.mean(embedding_latencies) if embedding_latencies else 0.0,
            "mean_search_latency_ms": statistics.mean(search_latencies) if search_latencies else 0.0,
            "p95_total_latency_ms": p95_latency,
        }

        return stats

    def get_performance_summary(self) -> Dict[str, Any]:
        """
        Returns comprehensive performance summary.

        Returns:
            Dict[str, Any]: Dictionary with:
                - 'total_queries': Total number of queries
                - 'cache_hit_rates': Cache hit rates
                - 'latency_stats': Latency statistics
                - 'error_rate': Error rate (if errors tracked)
        """
        total_queries = len(self.metrics_history)
        error_count = sum(1 for m in self.metrics_history if m.error_type is not None)
        error_rate = error_count / total_queries if total_queries > 0 else 0.0

        return {
            "total_queries": total_queries,
            "cache_hit_rates": self.get_cache_hit_rates(),
            "latency_stats": self.get_latency_stats(),
            "error_rate": error_rate,
        }

    def reset_metrics(self) -> None:
        """
        Clears all metrics history.
        """
        self.metrics_history.clear()

In [ ]:
"""Optimized search system integrating HNSW indexing, caching, and metrics collection.

This module provides a complete optimized search system that combines:
- HNSW indexing for fast approximate search
- Multi-level caching for performance
- Metrics collection for monitoring
"""

import time
import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


class OptimizedSearchSystem:
    """
    A complete optimized search system with HNSW indexing, caching, and monitoring.

    This system integrates:
    - Embedding generation (or uses VectorStore if available)
    - HNSWIndex: For fast approximate vector search
    - CacheManager: For multi-level caching
    - MetricsCollector: For performance monitoring

    The system processes queries with caching, uses HNSW for fast search, and tracks
    performance metrics for optimization.
    """

    def __init__(
        self,
        vector_store: VectorStore,
        hnsw_index: HNSWIndex,
        cache_manager: CacheManager,
        metrics_collector: MetricsCollector
    ):
        """
        Initializes the OptimizedSearchSystem with required components.

        Args:
            hnsw_index (HNSWIndex): HNSWIndex instance for fast search.
            cache_manager (CacheManager): CacheManager instance for caching.
            metrics_collector (MetricsCollector): MetricsCollector instance for monitoring.
            embedding_model (Optional[Any]): Optional embedding model (e.g., SentenceTransformer).
        """
        self.vector_store = vector_store
        self.hnsw_index = hnsw_index
        self.cache_manager = cache_manager
        self.metrics_collector = metrics_collector

    def search(
        self,
        query: str,
        k: int = 5,
        search_ef: int = 50,
        use_cache: bool = True
    ) -> Dict:
        """
        Main search method with caching, HNSW search, and metrics collection.

        Process:
        1. Check response cache (if enabled)
        2. Check search result cache (if enabled)
        3. Get query embedding (check embedding cache)
        4. Search HNSW index
        5. Cache results
        6. Record metrics
        7. Return results with performance stats

        Args:
            query (str): The user's search query.
            k (int): Number of results to return (default: 5).
            search_ef (int): HNSW search_ef parameter (default: 50).
            use_cache (bool): Whether to use caching (default: True).

        Returns:
            dict: Search results containing:
                - 'results': List of (node_id, distance) tuples
                - 'query_id': Unique query identifier
                - 'cache_hits': Dictionary of cache hit status
                - 'latency_ms': Total query latency
                - 'metrics': Performance metrics

        Raises:
            ValueError: If query is empty or k is invalid.
            TypeError: If query is not a string.
        """
        if not isinstance(query, str):
            raise TypeError("Query must be a string")

        if not query.strip():
            raise ValueError("Query must not be empty")

        if k < 1:
            raise ValueError("k must be at least 1")

        if search_ef < k:
            raise ValueError("search_ef must be greater than or equal to k")

        query_id = str(uuid.uuid4())
        cache_hits = {"embedding": False, "search": False, "response": False}
        start_time = time.perf_counter()

        # Step 1: Check response cache
        if use_cache:
            cached_response = self.cache_manager.get_response(query)
            if cached_response is not None:
                cache_hits["response"] = True
                total_latency = (time.perf_counter() - start_time) * 1000

                # Update the cached response with new cache_hits to reflect this is a cache hit
                updated_response = cached_response.copy()
                updated_response["cache_hits"] = cache_hits.copy()

                query_metric = QueryMetrics(
                    query_id=query_id,
                    timestamp=datetime.now(),
                    total_latency_ms=total_latency,
                    embedding_latency_ms=0.0,
                    search_latency_ms=0.0,
                    retrieved_count=len(cached_response.get("results", [])),
                    cache_hits=cache_hits,
                    error_type=None
                )
                self.metrics_collector.record_query(query_metric)
                return updated_response

        # Step 2: Check search result cache
        if use_cache:
            cached_results = self.cache_manager.get_search_results(query, k)
            if cached_results is not None:
                cache_hits["search"] = True
                total_latency = (time.perf_counter() - start_time) * 1000

                response = self._format_response(
                    cached_results, query_id, cache_hits, total_latency,
                    {"embedding_latency_ms": 0.0, "search_latency_ms": 0.0}
                )

                query_metric = QueryMetrics(
                    query_id=query_id,
                    timestamp=datetime.now(),
                    total_latency_ms=total_latency,
                    embedding_latency_ms=0.0,
                    search_latency_ms=0.0,
                    retrieved_count=len(cached_results),
                    cache_hits=cache_hits,
                    error_type=None
                )
                self.metrics_collector.record_query(query_metric)
                return response

        # Step 3: Get query embedding
        embedding_start = time.perf_counter()
        query_embedding, embedding_cache_hit = self._get_query_embedding(query, use_cache)
        embedding_latency_ms = (time.perf_counter() - embedding_start) * 1000
        cache_hits["embedding"] = embedding_cache_hit

        # Step 4: Search HNSW index
        search_start = time.perf_counter()
        results = self._search_hnsw(query_embedding, k, search_ef)
        search_latency_ms = (time.perf_counter() - search_start) * 1000

        # Step 5: Cache results
        if use_cache:
            self.cache_manager.set_search_results(query, k, results)

        # Step 6: Calculate total latency
        total_latency_ms = (time.perf_counter() - start_time) * 1000

        # Step 7: Format response
        metrics_data = {
            "embedding_latency_ms": embedding_latency_ms,
            "search_latency_ms": search_latency_ms
        }
        response = self._format_response(results, query_id, cache_hits, total_latency_ms, metrics_data)

        # Step 8: Cache response
        if use_cache:
            self.cache_manager.set_response(query, response)

        # Step 9: Record metrics
        query_metric = QueryMetrics(
            query_id=query_id,
            timestamp=datetime.now(),
            total_latency_ms=total_latency_ms,
            embedding_latency_ms=embedding_latency_ms,
            search_latency_ms=search_latency_ms,
            retrieved_count=len(results),
            cache_hits=cache_hits,
            error_type=None
        )
        self.metrics_collector.record_query(query_metric)

        return response

    def _get_query_embedding(self, query: str, use_cache: bool) -> Tuple[np.ndarray, bool]:
        """
        Gets query embedding with caching support.

        Args:
            query (str): Query string.
            use_cache (bool): Whether to use embedding cache.

        Returns:
            Tuple[np.ndarray, bool]: (embedding, cache_hit)
        """
        # Check embedding cache
        if use_cache:
            cached_embedding = self.cache_manager.get_embedding(query)
            if cached_embedding is not None:
                return cached_embedding, True

        if self.vector_store is not None:
            embedding = self.vector_store.get_query_embedding(query)
        else:
            # For testing: generate a random embedding
            # In production, this should use a real embedding model
            embedding = np.random.randn(384).astype(np.float32)
            embedding = embedding / np.linalg.norm(embedding)

        # Cache the embedding
        if use_cache:
            self.cache_manager.set_embedding(query, embedding)

        return embedding, False

    def _search_hnsw(self, query_embedding: np.ndarray, k: int, search_ef: int) -> List[Tuple[int, float]]:
        """
        Searches HNSW index for nearest neighbors.

        Args:
            query_embedding (np.ndarray): Query embedding vector.
            k (int): Number of results to return.
            search_ef (int): HNSW search_ef parameter.

        Returns:
            List[Tuple[int, float]]: List of (node_id, distance) tuples.
        """
        return self.hnsw_index.search(query_embedding, k=k, search_ef=search_ef)

    def _format_response(
        self,
        results: List[Tuple[int, float]],
        query_id: str,
        cache_hits: Dict[str, bool],
        latency_ms: float,
        metrics: Dict
    ) -> Dict:
        """
        Formats the final response with results and performance metrics.

        Args:
            results (List[Tuple[int, float]]): Search results.
            query_id (str): Query identifier.
            cache_hits (Dict[str, bool]): Cache hit status.
            latency_ms (float): Query latency.
            metrics (Dict): Performance metrics.

        Returns:
            dict: Formatted response dictionary.
        """
        return {
            "query_id": query_id,
            "results": results,
            "cache_hits": cache_hits,
            "latency_ms": latency_ms,
            "embedding_latency_ms": metrics.get("embedding_latency_ms", 0.0),
            "search_latency_ms": metrics.get("search_latency_ms", 0.0),
            "retrieved_count": len(results),
            "metrics": metrics
        }